In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.formula.api import logit
from scipy.stats import chi2_contingency, chi2

from pathlib import Path
import pickle
import os

#### Project Question
* Does region, payment method, age, or gender have statistical association with customer churn?

In [2]:
# Establishing directory and file paths
project_dir = Path.cwd()
parent_dir = Path.cwd().parent
dataset = f'{parent_dir}/data/customer_churn_clean.csv'
results_dir = f'{parent_dir}/results/'

# Importing dataset
df=pd.read_csv(dataset, index_col=0)
df.head()

,customer_id,gender,region,subscription_plan,payment_method,age,days_since_last_login,customer_service_calls,monthly_spend,churn
0,10001,male,east,basic,paypal,27.0,27,1.0,61.08,0
1,10002,unknown,west,pro,paypal,55.0,146,2.0,73.09,0
2,10003,female,west,pro,e-wallet,NaN,147,2.0,116.30,1
3,10004,female,south,basic,credit_card,NaN,34,1.0,0.58,1
4,10005,male,north,basic,crypto,30.0,146,3.0,46.35,1


In [3]:
# Dataset Information
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10300 entries, 0 to 10299
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   customer_id             10300 non-null  int64  
 1   gender                  10300 non-null  str    
 2   region                  10300 non-null  str    
 3   subscription_plan       10300 non-null  str    
 4   payment_method          9575 non-null   str    
 5   age                     8718 non-null   float64
 6   days_since_last_login   10300 non-null  int64  
 7   customer_service_calls  10248 non-null  float64
 8   monthly_spend           9715 non-null   float64
 9   churn                   10300 non-null  int64  
dtypes: float64(3), int64(3), str(4)
memory usage: 804.8 KB


#### Variables for Analysis
* Target Variable: Churn
    - 1 = Churned
    - 0 = Stayed

* Predictor Variables
    - Region
        - Geographic Territory
    - Gender
        - Reported Customer Gender
    - Payment Method
        - Most Frequent Payment Type
    - Age
        - Customer's Age in Years

#### Objectives
- Identify if there's a statistically significant association between target variable and predictor variables
- Quantify the Odds, Odds Ratio, and Probability of statistically significant variables

**_Methods_**
* Chi-Squared ($\chi^2$) Test of Independence
* Logistic Regression between Churn and Age
* Likelihood Ratio using Chi-Squared Test ($\chi^2$)
* Logistic Regression on Churn Asociated Variables

In [4]:
# File saving function
os.makedirs(results_dir,exist_ok=True)
def save_result(path:str,filename:str,data):
    with open(path+filename+'.pkl', 'wb') as to_write:
        pickle.dump(data,to_write)

# Read saved results function
def read_result(path:str,filename:str):
    with open(path+filename+'.pkl', 'rb') as to_read:
        result = pickle.load(to_read)
        return result

# Chi Test (Isolated and Multiple)
def chi_test(columns,data,target,alpha=0.05,results_dir=None,test_name=None):
    rows=[]
    for col in columns:
        observed=pd.crosstab(data[target],data[col])
        res = chi2_contingency(observed)

        rows.append({
            'Model':f'{target} ~ {col}',
            'Statistics':(res.statistic).round(2),
            'ddof':res.dof,
            'P Value':(f'{res.pvalue:.10f}'),
            'Reject':res.pvalue<alpha
        })
        results=pd.DataFrame(rows)

    if results_dir and test_name:
        save_result(results_dir,test_name,results)

# Logistic modeling
def model_logit(formula:str,data):
    return logit(formula,data).fit(disp=False)

# Logistic Summary Table
def model_table(model):
    return model.summary().tables[1]

# Chi-Squared Test for Controlled Variables
def controlled_chi_test(controls,predictor,target,data,alpha=0.05):
    full = model_logit(f'{target} ~ {predictor} + {controls}', data)
    used = data.loc[full.model.data.row_labels]              # identical rows for both models
    reduced = model_logit(f'{target} ~ {controls}', used)
    if int(full.nobs) != int(reduced.nobs):
        raise ValueError('Models were fit on different rows.')

    lr = 2 * (full.llf - reduced.llf)
    dof = int(full.df_model - reduced.df_model)
    p = chi2.sf(lr, dof)
    return pd.DataFrame({
        'N': int(full.nobs), 'LR': round(lr, 2), 'DoF': dof,
        'P Value': p, 'Reject': p < alpha,
    }, index=[predictor])

# Variable Name Cleaning
def clean_name(model):
    return model.str.replace(
        r'C\(([^)]+)\)\[T\.(.*?)\]',
        lambda a: f'{a.group(1).capitalize()} ~ {a.group(2).capitalize()}',
        regex=True
    )

# Logistic Model Controlled Odds
def explain_model(model,ref):
    intercept = model.params.iloc[0]
    params = model.params.drop('Intercept')

    inter = pd.DataFrame({
        'Odds':np.exp(intercept),
        'Churn Probability':np.exp(intercept)/(1+np.exp(intercept)),
        'Odds Ratio':1,
        'Gap':'reference'
    }, index=[f'{ref}'.capitalize()])

    rest = pd.DataFrame({
        'Odds':np.exp(intercept+params),
        'Churn Probability':np.exp(intercept+params)/(1+np.exp(intercept+params)),
        'Odds Ratio':np.exp(params),
        'Gap':np.exp(intercept+params)/(1+np.exp(intercept+params)) - (np.exp(intercept)/(1+np.exp(intercept)))
    })

    return pd.concat([inter,rest]).reset_index().sort_values(by='Churn Probability')

# Predicted Churn Probability on Controlled Variables
def predict_churn_prob(model, data, var='region'):
    rows = []
    for level in sorted(data[var].unique()):
        tmp = data.copy()
        tmp[var] = level                              # change region only for prediction
        rows.append({
            var.title(): level,
            'Adjusted Churn Probability': model.predict(tmp).mean(),
        })
    return (pd.DataFrame(rows)
            .sort_values('Adjusted Churn Probability')
            .reset_index(drop=True))

def cramers_v(table):
    stat = chi2_contingency(table).statistic
    return np.sqrt(stat / (table.to_numpy().sum() * (min(table.shape) - 1)))

##### *Conducting Chi-Squared ($\chi^2$) Test of Independence*
>**_Hypothesis Test_**
>
>Variable | $H_0$ | $H_1$
>---|---|---
>Region | Customer churn is independent of region | Customer churn depends on region
>Gender | Customer churn is independent of region | Customer churn depends on region
>Payment Method | Customer churn is independent of region | Customer churrn >depends on region

##### *Conducting Isolated Logistic Regression*

>**_Hypothesis Test_**
>
>$H_0:$ Customer's Age is independent to Churn
>
>$H_1:$ Customer's Age is dependent to Churn

>---
#### _Chi Squared ($\chi^2$) Test_
>Target Variable
>* Churn
>
>Predictor Variables:
>* Region
>* Gender
>* Payment Method

>---
#### _Logistic Regression_
>Target Variable
>* Churn
>
>Predictor Variable:
>* Age

>---

In [5]:
df_test = df[df.columns[1:]].dropna().copy()
df_test

,gender,region,subscription_plan,payment_method,age,days_since_last_login,customer_service_calls,monthly_spend,churn
0,male,east,basic,paypal,27.0,27,1.0,61.08,0
1,unknown,west,pro,paypal,55.0,146,2.0,73.09,0
4,male,north,basic,crypto,30.0,146,3.0,46.35,1
5,male,west,basic,crypto,46.0,103,1.0,36.64,1
6,male,south,basic,e-wallet,22.0,62,2.0,0.00,1
...,...,...,...,...,...,...,...,...,...
10293,male,south,pro,crypto,45.0,0,1.0,29.04,1
10294,male,east,basic,paypal,51.0,69,3.0,14.19,1
10295,female,west,pro,crypto,31.0,149,1.0,164.57,1
10296,female,north,plus,crypto,40.0,112,2.0,177.55,1


In [6]:
# Set variables for chi squared test
cols = df.columns[1:]
controls = ['region','gender','payment_method']

# Set target variable
# test name
# alpha
target = 'churn'
chi2_test_name = 'chi_square_test_results'
confidence_level = 0.05

# Get chi test results
chi_test_result = chi_test(
    controls, # columns
    df_test, # data
    target, # target
    confidence_level, # alpha
    results_dir, # directory for saving
    chi2_test_name # results file name
)
result = read_result(results_dir,chi2_test_name)
result

,Model,Statistics,ddof,P Value,Reject
0,churn ~ region,32.95,3,0.0000003295,True
1,churn ~ gender,0.16,3,0.9839990715,False
2,churn ~ payment_method,7.00,3,0.0718577490,False


In [7]:
# Get isolated logistic regression of churn and age
model = model_logit('churn ~ age',df_test)
summary = model_table(model)
summary

,coef,std err,z,P>|z|,[0.025,0.975]
Intercept,1.1101,0.107,10.420,0.000,0.901,1.319
age,0.0010,0.003,0.326,0.744,-0.005,0.007


_Chi-Squared (χ²) Test Results_

| Variable | Conclusion | Result |
|---|---|---|
| Region | Statistically significant association with churn | χ² = 32.95, df = 3, p < 0.0001 |
| Gender | No statistically significant association with churn | χ² = 0.16, df = 3, p = 0.98 |
| Payment method | No statistically significant association with churn (borderline) | χ² = 7.00, df = 3, p = 0.072 |

_Churn ~ Age Logistic Regression_
* Fail to reject $H_0$: there is no statistically significant association between churn and age (z = 0.33, p = 0.744).

##### _Next Steps_
_Likelihood Ratio using Chi-Squared ($\chi^2$) Test of Independence under Controlled Variables_

>Target Variable:
>- Churn
>
>Predictor Variables:
>- Region
>- Monthly Spend
>- Customer Service Calls
>- Days Since Last login
>- Age

$H_0:$ Region has no statistically significant association with customer churn after controlling for other variables.

$H_1:$ Region has statistically significant association with customer churn after controlling for other variables.

In [8]:
# Specify the columns
cols = ['region','monthly_spend','customer_service_calls','days_since_last_login','age','churn']
controls = 'monthly_spend + customer_service_calls + days_since_last_login + age'
predictor = 'region'
target = 'churn'
controlled_chi_test(
    controls,
    predictor,
    target,
    df[cols].dropna().copy()
)

,N,LR,DoF,P Value,Reject
region,8168,47.48,3,2.750162e-10,True


**_Conclusion_**
* Reject $H_0$ and conclude that region has statistically significant association with customer churn after controlling for other variables

**_Next Steps_**
* Perform an isolated logistic regression between region and churn
    - Gender, Payment Method, and Age are omitted from the model which are non-significant on customer churn

In [9]:
formula = 'churn ~ C(region)'
inf_model = model_logit(formula, df_test)
inf_result = explain_model(inf_model,'East')
inf_result['index'] = clean_name(inf_result['index'])
inf_result

,index,Odds,Churn Probability,Odds Ratio,Gap
0,East,2.503704,0.714588,1.000000,reference
1,Region ~ North,3.097130,0.755927,1.237019,0.041339
3,Region ~ West,3.358447,0.770561,1.341392,0.055973
2,Region ~ South,3.794554,0.791430,1.515576,0.076842


**_Results_**
* All regions show an unusually high churn rate, with a minimum of `71.5%` (East).
* Compared with East:
    - North is 4.1 percentage points higher (75.6%)
    - West is 5.6 points higher (77.1%)
    - South is 7.7 points higher (79.1%)
* The step between regions is uneven: East → North +4.1, North → West +1.5, West → South +2.1 points.
* Region effect size: Cramér's V = 0.066, a very weak association.

In [10]:
formula = 'churn ~  C(region) + ' + controls
model = model_logit(formula, df_test)
summary = model_table(model)
summary

,coef,std err,z,P>|z|,[0.025,0.975]
Intercept,-0.2620,0.143,-1.831,0.067,-0.542,0.018
C(region)[T.north],0.2522,0.080,3.136,0.002,0.095,0.410
C(region)[T.south],0.4884,0.081,6.003,0.000,0.329,0.648
C(region)[T.west],0.3071,0.080,3.818,0.000,0.149,0.465
monthly_spend,-0.0069,0.000,-14.503,0.000,-0.008,-0.006
customer_service_calls,0.3654,0.024,15.348,0.000,0.319,0.412
days_since_last_login,0.0139,0.001,23.224,0.000,0.013,0.015
age,-0.0003,0.003,-0.102,0.918,-0.007,0.006


##### Quantifying the Effect of Region 

In [11]:
print('Region Effect:',cramers_v(pd.crosstab(df_test['churn'], df_test['region'])))

Region Effect: 0.06587365873937441
